# Automatsko prepoznavanje akorda

## 1) Problem i dataset (GuitarSet)
- **Problem**: dati audio → vratiti niz **akorda** po vremenu (sekvencijalna klasifikacija).
- **Ulaz/izlaz**: ulaz `wav/mp3`, izlaz `[t0, t1, chord]` segmenti.
- **Skup podataka**: **GuitarSet** (360 isečaka × 30s), anotacije su u **JAMS** formatu, audio u `data/audio_mono-pickup_mix`, anotacije u `data/annotation`.
- **Vokabular**: dur/mol trijade + posebna klasa **N** (no chord).

Praktične primene: edukacija, transkripcija, muzička pretraga...


## 1.1 GuitarSet
- Sirovi audio: podfolder **`data/audio_mono-pickup_mix`** (mono mikseta).
- Anotacije: **JAMS** fajlovi u **`data/annotation`** — sadrže vremenski poravnate akorde.
- Spajanje parova: oslanjamo se na **isti stem** imena, uz napomenu da audio fajl može imati sufiks `_mix`.
- **Vokabular** je mapiran u studentskoj varijanti: osnovne trijade (dur/mol) + `N` (nema akorda).

## 2) Kako projekat radi
1. **Učitavanje & normalizacija** audio signala (`librosa`).
2. **Ekstrakcija fičra**: **chroma CQT** (12 hroma kanala).
3. **Beat sinhronizacija**: agregacija fičra po bitovima radi stabilnijih etiketa.
4. **Mapiranje akorda** iz JAMS → u trijade (dur/mol) + `N`.
5. **Modeli**:
   - **HMM baseline** (GaussianHMM nad beat-sinhr. hromom)
   - **CNN** (2D konvolucije nad hromogramom → proj na klase)
   - **LSTM** (sekvencijalne zavisnosti; koristi `pack_padded_sequence`)
   - **CNN+LSTM** (konvolucije za lokalne šare + LSTM za tok)
6. **Metrike**: **CSR**, **WCSR**, **Overlap** (trajanje-ponderisana tačnost i preklapanje).


## 2.1 Šta tačno rešavamo
- **Ulaz**: audio signal pesme (`.wav`/`.mp3`), ne moramo znati tempo/tonalitet unapred.
- **Izlaz**: niz segmenata *[početak, kraj, labela_akorda]* gde je labela iz definisanog vokabulara (dur/mol trijade + `N`).
- **Zašto beat‑sinhronizacija?** Akordi se najčešće menjaju oko bitova/taktova → stabilnije etikete i manje šuma nego po freamovima.
- **Zašto chroma?** Chroma sabija spektar u 12 klasa visina.

## 3) Setup (uvoz modula i putanja)

## 3.1 Pretprocesiranje i fičri
**Audio koraci**
- učitavanje i, po potrebi, resamplovanje na `SR` (vidi `feature_extraction.SR`), normalizacija amplitude.

**Chroma CQT**
- `librosa.feature.chroma_cqt(...)` → matrica oblika **12×T**.

**Beat‑sync**
- detekcija bitova i dobijanje **granica** (*boundaries*).

In [ ]:
import warnings, sys
from pathlib import Path
warnings.filterwarnings(
    "ignore",
    message=r"pkg_resources is deprecated as an API.*",
    category=UserWarning,
)

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT/"src"
DATA = PROJECT_ROOT/"data"
ANNO = DATA/"annotation"
AUDIO = DATA/"audio_mono-pickup_mix"
assert SRC.exists(), f"Ne nalazi se src/: {SRC}"
sys.path.insert(0, str(SRC))

from utils import find_pairs
from feature_extraction import load_audio, chroma_cqt, beat_sync, SR
from data_utils import extract_pair
from chord_vocab import id_to_chord_label
from hmm_baseline import decode_song, load_model as load_hmm
print("Project root:", PROJECT_ROOT)
print("Data exists:", ANNO.exists(), AUDIO.exists())

## 4) Pregled podataka i jedan primer fičra
Uzorak: učitavanje jednog para (JAMS + WAV), ekstrakcija **chroma** i **beat** granica, i kratko crtanje hromograma.

In [ ]:
import random
import matplotlib.pyplot as plt

pairs = find_pairs()
print(f"Ukupno upareno: {len(pairs)}")
jams_path, wav_path = random.choice(pairs)
print("Primer:", wav_path.name)

y, sr = load_audio(str(wav_path), sr=SR)
C = chroma_cqt(y, sr)
X_sync, boundaries = beat_sync(C, y, sr)

plt.figure(figsize=(8,3))
plt.imshow(C, aspect='auto', origin='lower')
plt.title('Chroma CQT (primer)')
plt.xlabel('frame'); plt.ylabel('chroma bin')
plt.show()

print("Beat-segmenata:", X_sync.shape[1], "| Trajanje sekvenci u s:", boundaries[-1]-boundaries[0])

## 5) HMM baseline - treniranje i evaluacija

## 5.1 Metrike
- **CSR** (*Chord Symbol Recall*): udeo tačno pogođenih beat‑ova.
- **WCSR**: CSR ponderisan **trajanjem** (duži segmenti imaju veću težinu).
- **Overlap**: mera preklapanja tačnih/prediktovanih segmenata (takođe po trajanju).

**Praktično**: ako WCSR ~ 0.30-0.35 za baseline na krnjem treningu → realno za demo

In [ ]:
from actions import action_train_hmm, action_evaluate_hmm
from utils import MODELS_DIR
hmm_path = MODELS_DIR/"hmm_baseline.pkl"
if not hmm_path.exists():
    print("Trening HMM (skracen skup)...")
    action_train_hmm()
else:
    print("Postojeći HMM model je pronađen:", hmm_path)

print("Evaluacija HMM:")
action_evaluate_hmm()

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

def load_eval_json(p: Path):
    if p.exists():
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            return None
    return None

OUT = Path("outputs")
hmm = load_eval_json(OUT/"eval_hmm.json")
cnn = load_eval_json(OUT/"eval_cnn.json")

def plot_metric(eval_dict, title):
    if not eval_dict:
        print(title, ": (nema podataka, pokreni evaluaciju)")
        return
    splits = []
    csr = []; wcsr = []; ov = []
    for s in ["train","val","test"]:
        if s in eval_dict:
            splits.append(s)
            csr.append(eval_dict[s].get("CSR",0))
            wcsr.append(eval_dict[s].get("WCSR",0))
            ov.append(eval_dict[s].get("Overlap",0))
    if not splits:
        print(title, ": (nema splitova)"); return
    import numpy as np
    x = np.arange(len(splits))
    plt.figure(figsize=(6,3.2))
    plt.plot(x, csr, marker='o', label='CSR')
    plt.plot(x, wcsr, marker='o', label='WCSR')
    plt.plot(x, ov, marker='o', label='Overlap')
    plt.xticks(x, splits)
    plt.title(title)
    plt.xlabel("Split"); plt.ylabel("Score")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_metric(hmm, "HMM metrics")
plot_metric(cnn, "CNN metrics")

## 6) NN modeli (CNN / LSTM / CNN+LSTM)

## 6.1 Modeli
**HMM (baseline)**
- **Stanja** = klase akorda; **emisije** = Gausovi nad 12‑D vektorima (po beat‑u).
- Učenje: računamo srednje vrednosti i kovarijanse po klasi + tranzicije (bigrami akorda).
- Dekodiranje: **Viterbi** (najverovatniji niz stanja). **Prednost**: jeftino, lako objašnjivo. **Mane**: ograničen kapacitet.

**CNN**
- Gleda hromogram kao *sliku* (frekvencija×vreme) → lokalne filtere po 2D.
- Dobar u hvatanju kratkih “otisaka” akorda; često najstabilniji od NN u studentskoj postavci.

**LSTM**
- Modeluje **sekvencijalne** odnose (prelazi akorda, trajanja) kroz vreme.
- U ovoj postavci koristimo `pack_padded_sequence` da ignorišemo padding.
- Korisno kad imamo duže kontinualne zapise i više varijacija progresija.

**CNN+LSTM**
- CNN za lokalne šare → LSTM za duži kontekst.
- Često radi najbolje kada ima dovoljno podataka/epoha.

## 7) Demo predikcije na nasumičnoj pesmi

In [ ]:
pairs = find_pairs()
jams_path, wav_path = random.choice(pairs)
print("Target za demo:", wav_path.name)
sample = extract_pair(jams_path, wav_path, cache=True)

model = load_hmm(hmm_path)
y_pred = decode_song(model, sample.X)
lines = []
T = min(len(y_pred), len(sample.boundaries)-1)
for b in range(T):
    t0, t1 = sample.boundaries[b], sample.boundaries[b+1]
    chord = id_to_chord_label(int(y_pred[b]))
    lines.append((t0, t1, chord))

print("HMM - prvih 15 segmenata:")
for row in lines[:15]:
    print(f"{row[0]:6.2f}-{row[1]:6.2f}  {row[2]}")